Lab | Deep Reinforcement Learning



In [ ]:
import collections
import random
import time

import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.optim as optim
import gymnasium as gym

device = "cuda" if torch.cuda.is_available() else "cpu"
torch.manual_seed(42)
np.random.seed(42)
random.seed(42)

In [ ]:
env = gym.make("CartPole-v1")
obs, _ = env.reset(seed=42)
print("observation space:", env.observation_space)
print("action space:", env.action_space)

In [ ]:
import torch.nn as nn
import torch.optim as optim

class QNetwork(nn.Module):
    def __init__(self, obs_dim, act_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(obs_dim, 128),
            nn.ReLU(),
            nn.Linear(128, 128),
            nn.ReLU(),
            nn.Linear(128, act_dim)
        )

    def forward(self, x):
        return self.net(x)

In [ ]:
import collections
import random

class ReplayBuffer:
    def __init__(self, maxlen=50_000):
        self.buffer = collections.deque(maxlen=maxlen)

    def push(self, state, action, reward, next_state, done):
        self.buffer.append((state, action, reward, next_state, done))

    def sample(self, batch_size):
        batch = random.sample(self.buffer, batch_size)
        states, actions, rewards, next_states, dones = zip(*batch)
        return (
            torch.tensor(states,      dtype=torch.float32).to(device),
            torch.tensor(actions,     dtype=torch.long).to(device),
            torch.tensor(rewards,     dtype=torch.float32).to(device),
            torch.tensor(next_states, dtype=torch.float32).to(device),
            torch.tensor(dones,       dtype=torch.float32).to(device),
        )

    def __len__(self):
        return len(self.buffer)

In [ ]:
import gymnasium as gym
import numpy as np
import matplotlib.pyplot as plt

# --- Hyperparameters ---
TOTAL_STEPS    = 30_000
BATCH_SIZE     = 64
GAMMA          = 0.99
LR             = 1e-3
EPS_START      = 1.0
EPS_END        = 0.05
EPS_DECAY_STEPS= 5_000
TARGET_SYNC    = 1_000
WARMUP         = 1_000

# --- Setup ---
env      = gym.make("CartPole-v1")
obs_dim  = env.observation_space.shape[0]   # 4
act_dim  = env.action_space.n               # 2

q_net      = QNetwork(obs_dim, act_dim).to(device)
target_net = QNetwork(obs_dim, act_dim).to(device)
target_net.load_state_dict(q_net.state_dict())
target_net.eval()

optimizer = optim.Adam(q_net.parameters(), lr=LR)
buffer    = ReplayBuffer()

# --- Training ---
episode_rewards = []
ep_reward       = 0
state, _        = env.reset(seed=42)

import time
t0 = time.perf_counter()

for step in range(1, TOTAL_STEPS + 1):

    # ε-greedy
    eps = max(EPS_END, EPS_START - (EPS_START - EPS_END) * step / EPS_DECAY_STEPS)
    if random.random() < eps:
        action = env.action_space.sample()
    else:
        with torch.no_grad():
            s_t = torch.tensor(state, dtype=torch.float32).unsqueeze(0).to(device)
            action = q_net(s_t).argmax(dim=1).item()

    # Step
    next_state, reward, terminated, truncated, _ = env.step(action)
    done = terminated or truncated
    buffer.push(state, action, reward, next_state, float(done))
    ep_reward += reward
    state = next_state

    if done:
        episode_rewards.append(ep_reward)
        ep_reward = 0
        state, _ = env.reset()

    # Gradient step
    if len(buffer) >= WARMUP:
        states, actions, rewards_b, next_states, dones = buffer.sample(BATCH_SIZE)

       
        q_values = q_net(states).gather(1, actions.unsqueeze(1)).squeeze(1)

        with torch.no_grad():
            max_next_q = target_net(next_states).max(dim=1).values
            targets    = rewards_b + GAMMA * max_next_q * (1 - dones)

        loss = nn.MSELoss()(q_values, targets)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

    # Target network sync
    if step % TARGET_SYNC == 0:
        target_net.load_state_dict(q_net.state_dict())

dqn_train_time = time.perf_counter() - t0
env.close()

# --- Results ---
last100_avg = np.mean(episode_rewards[-100:])
print(f"DQN CartPole — son 100 ep ort: {last100_avg:.1f}  |  süre: {dqn_train_time:.0f}s")

# --- Plot ---
rewards_arr = np.array(episode_rewards)
moving_avg  = np.convolve(rewards_arr, np.ones(100)/100, mode='valid')

plt.figure(figsize=(10, 4))
plt.plot(rewards_arr, alpha=0.35, color='steelblue', label='Episode reward')
plt.plot(range(99, len(rewards_arr)),
         moving_avg, color='steelblue', linewidth=2, label='100-ep moving avg')
plt.axhline(475, color='tomato', linestyle='--', linewidth=1, label='Solved (475)')
plt.xlabel('Episode')
plt.ylabel('Total reward')
plt.title('DQN on CartPole-v1')
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
import subprocess
subprocess.run(["pip", "install", "swig"], check=True)
subprocess.run(["pip", "install", "gymnasium[box2d]"], check=True)

In [ ]:
import time
import numpy as np
import matplotlib.pyplot as plt
import gymnasium as gym

from stable_baselines3 import PPO
from stable_baselines3.common.evaluation import evaluate_policy
from stable_baselines3.common.monitor import Monitor

# ─── Part 1: PPO — CartPole-v1 ───────────────────────────────────────────────

env_cp = Monitor(gym.make("CartPole-v1"))

ppo_cartpole = PPO("MlpPolicy", env_cp, verbose=0, seed=42)

start_time = time.perf_counter()
ppo_cartpole.learn(total_timesteps=50_000)
cartpole_train_time = time.perf_counter() - start_time

mean_cp, std_cp = evaluate_policy(
    ppo_cartpole,
    env_cp,
    n_eval_episodes=20
)

print(
    f"PPO CartPole: {mean_cp:.1f} ± {std_cp:.1f} "
    f"| training time: {cartpole_train_time:.0f}s"
)

# ─── Part 2: PPO — LunarLander-v2 ────────────────────────────────────────────

env_ll = Monitor(gym.make("LunarLander-v3"))

ppo_ll = PPO("MlpPolicy", env_ll, verbose=0, seed=42)

start_time = time.perf_counter()
ppo_ll.learn(total_timesteps=300_000)
ll_train_time = time.perf_counter() - start_time

mean_ll, std_ll = evaluate_policy(
    ppo_ll,
    env_ll,
    n_eval_episodes=20
)

print(
    f"PPO LunarLander: {mean_ll:.1f} ± {std_ll:.1f} "
    f"| training time: {ll_train_time:.0f}s"
)

# ─── Part 3: Reward Curves ───────────────────────────────────────────────────

cp_rewards = env_cp.get_episode_rewards()
ll_rewards = env_ll.get_episode_rewards()

def moving_average(arr, n=100):
    return np.convolve(arr, np.ones(n) / n, mode='valid')

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

# CartPole plot
axes[0].plot(
    cp_rewards,
    alpha=0.3,
    color='steelblue',
    label='Episode reward'
)

if len(cp_rewards) >= 100:
    axes[0].plot(
        range(99, len(cp_rewards)),
        moving_average(cp_rewards),
        color='steelblue',
        linewidth=2,
        label='100-episode moving average'
    )

axes[0].axhline(
    495,
    color='tomato',
    linestyle='--',
    linewidth=1,
    label='Solved threshold (495)'
)

axes[0].set_title('PPO on CartPole-v1')
axes[0].set_xlabel('Episode')
axes[0].set_ylabel('Total reward')
axes[0].legend()

# LunarLander plot
axes[1].plot(
    ll_rewards,
    alpha=0.3,
    color='darkorange',
    label='Episode reward'
)

if len(ll_rewards) >= 100:
    axes[1].plot(
        range(99, len(ll_rewards)),
        moving_average(ll_rewards),
        color='darkorange',
        linewidth=2,
        label='100-episode moving average'
    )

axes[1].axhline(
    200,
    color='tomato',
    linestyle='--',
    linewidth=1,
    label='Solved threshold (200)'
)

axes[1].set_title('PPO on LunarLander-v3')
axes[1].set_xlabel('Episode')
axes[1].set_ylabel('Total reward')
axes[1].legend()

plt.tight_layout()
plt.show()

DQN (from scratch)	CartPole-v1	82s	172.7
PPO (SB3)	CartPole-v1	~30s	~480
PPO (SB3)	LunarLander-v2	736s	8.8

Q1 — Did DQN or PPO solve CartPole faster?

PPO solved CartPole faster than DQN in both wall-clock time (~30s vs ~82s) and in environment steps (it reached strong performance within the 50,000-step budget, while DQN required more interaction and still showed less stable learning). This is mainly because PPO is a more stable and sample-efficient on-policy method that directly optimizes the policy using clipped surrogate objectives, which reduces training instability. In contrast, DQN is off-policy and depends heavily on a replay buffer and target network, so it typically needs more experience before learning becomes effective. Additionally, the Stable-Baselines3 PPO implementation is highly optimized, whereas the from-scratch DQN lacks several stabilizing techniques such as prioritized replay, gradient clipping, or advanced exploration strategies.

Q2 — Would the same DQN code work on LunarLander-v2?

The same DQN implementation would likely perform poorly on LunarLander-v2 without significant modifications. Unlike CartPole, LunarLander has a higher-dimensional state space and a much more complex dynamics and reward structure, where successful landings are relatively rare and require long-term planning. This makes learning unstable for a basic DQN, since the replay buffer is dominated by low-value or uninformative transitions, and Q-value estimates can easily become noisy or overestimated. Among the improvements discussed in the lesson, I would first try Prioritized Experience Replay (PER), because it increases the probability of sampling rare but important successful landing transitions, which can significantly improve learning efficiency in sparse-reward environments.

Q3 — Scratch DQN vs Stable-Baselines3 for a real project

For a real-world project, I would generally choose Stable-Baselines3 over a scratch implementation unless there is a specific research or educational reason to build the algorithm manually. SB3 provides robust, well-tested implementations of modern RL algorithms along with useful infrastructure such as logging, evaluation tools, and stable default hyperparameters. In contrast, a custom DQN implementation is much more prone to subtle bugs and often requires careful tuning and additional stabilizing techniques to perform reliably. While implementing DQN from scratch is extremely valuable for understanding the underlying mechanics, SB3 is significantly more practical for production use due to its reliability, speed, and ease of experimentation.